# 四数据集 Ground-truth Ledger

本 Notebook 从项目现有 canonical sample provider 与 manifest 建立 M-Unified 原始标签账本；不改写 `dataset/raw` 或 `dataset/processed`。四个数据集继续保留各自 prediction unit：ICBHI cycle、SPRSound event、HF 2秒source-time window、KAUH recording，因此support只在各数据集内部报告，不做虚假跨数据集求和。

## Ontology 与 eligibility 边界

- ICBHI：flat4精确映射到Level1、Crackle、Wheeze；Other为显式0。
- SPRSound：Normal、Coarse/Fine Crackle、Wheeze、Wheeze+Crackle为compatible subset；Rhonchi/Stridor仅标Level1异常和Proposed Other阳性，Crackle/Wheeze保持Unknown mask。
- HF：保留I/E/D/Wheeze/Rhonchi/Stridor原始区间；窗口中心命中D、Wheeze、Rhonchi/Stridor时仅产生对应阳性监督。Level1、gap、empty与未命中通道全部mask，绝不构造negative。
- KAUH：只映射N、E W、I E W、C、I C、I C E W；Crep、Bronchial、I C B继续HOLD。B/D/E仍按同一P-number分组。

SPRSound inter test标签在validation选模完成、label-free预测写出之后才接入最终test ledger。

In [ ]:
from pathlib import Path
import json
from baseline.multidataset_pipeline.m_unified import build_ground_truth_ledger

ROOT = Path.cwd()
OUTPUT = ROOT / 'result/reproduce/unified/ground_truth_ledger'
manifest_path = OUTPUT / 'ledger_manifest.json'
manifest = json.loads(manifest_path.read_text()) if manifest_path.is_file() else build_ground_truth_ledger(ROOT, OUTPUT)
manifest

In [ ]:
support = json.loads((OUTPUT / 'support_summary.json').read_text())
support

输出写入 `result/reproduce/unified/ground_truth_ledger/`。`ledger_test_manifest.jsonl` 中SPRSound terminal rows最初保持mask；完整训练选模后由01 Notebook对应运行补写 `ledger_test.jsonl` 与最终support summary。